# Decorte + LSTM model — v3 (skills concatenated at output, last-hidden-state pooling)


Install all necessary packages to run and analyze this model

In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install matplotlib pyyaml datasets pandas tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


Check if we run on CUDA

In [3]:
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand(3, 3).to(device)
print(x.device)  # Should output 'cuda:0'

True
cuda:0


Let's do a set up

In [4]:
dataset_name = "decorte"
vector_transformation_config = f"{dataset_name}.yaml"
from pathlib import Path

DATA_PATH = Path("./data/")

Load the config

In [5]:
import os
import yaml
import json

with open(os.path.join("config/train/", vector_transformation_config)) as file:
    config = yaml.safe_load(file)

# Load default neural config
neural_default_path = os.path.join("config/train/", "neural_transformation.yaml")
if os.path.exists(neural_default_path):
    with open(neural_default_path) as file:
        neural_defaults = yaml.safe_load(file).get("neural", {})
    # Merge defaults into config['neural']
    if "neural" not in config:
        config["neural"] = neural_defaults
    else:
        # Fill in missing keys from defaults
        for k, v in neural_defaults.items():
            if k not in config["neural"]:
                config["neural"][k] = v

print("Loaded configuration:")
print(json.dumps(config, indent=4))

Loaded configuration:
{
    "model": {
        "embedding_model_finetuning": "all-mpnet-base-v2",
        "embedding_model_transformation": "ElenaSenger/career-path-representation-mpnet-decorte"
    },
    "data": {
        "data_type": "decorte"
    },
    "output": {
        "path_embedding_model": "./output/all-mpnet-base-v2_finetuned_decorte",
        "path_neural_transformation_model": "./output/vector_transform_model_decorte.pth"
    },
    "neural": {
        "batch_size": 256,
        "learning_rate": 2e-05,
        "epochs": 50,
        "patience": 6,
        "hidden_sizes": [
            512
        ],
        "dropout": true,
        "dropout_rate": 0.5
    }
}


Load the dataset.

In [6]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# Load pre-trained sentence transformer model
model = SentenceTransformer(config["model"]["embedding_model_transformation"])

# Data
### Load data for neural transformation training
print("Loading data...")

# Load the dataset from local CSVs (same schema as the original HF dataset).
DECORTE_DIR = DATA_PATH / "decorte-extended"
dataset = load_dataset(
    "csv",
    data_files={
        "train":      str(DECORTE_DIR / "train.csv"),
        "validation": str(DECORTE_DIR / "validation.csv"),
        "test":       str(DECORTE_DIR / "test.csv"),
    },
)

/home/nikita/projects/kpi-mag-models-extended/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3602.27it/s]


Loading data...


Replace some ESCO titles.

In [7]:
import pandas as pd

def replace_esco_titles(example, i):
    """
    Replaces specific ESCO job titles with alternative titles for consistency.

    Args:
        example (dict): A dictionary representing a dataset row.
        i (int): The index of the ESCO title column.

    Returns:
        dict: Updated dictionary with the replaced ESCO title and URI.
    """
    replacements_title = {
        'ICT security engineer': 'cyber incident responder',
        'ict security engineer': 'cyber incident responder',
        'care at home worker': 'care home worker',
        'residential care home worker': 'care home worker',
        'ICT security manager': 'cybersecurity risk manager',
        'ict security manager': 'cybersecurity risk manager',
        'care at hmoe worker': 'care home worker',
        'handyman': 'handyperson',
        'corporate banking manager': 'corporate banking adviser',
    }

    original_title = example[f'ESCO_title_{i}']
    if not pd.isna(original_title):
        processed_title = original_title.strip().lower()
        final_title = replacements_title.get(processed_title, processed_title)
    else:
        final_title = original_title

    example[f'ESCO_title_{i}'] = final_title

    replacements_uri = {
        'http://data.europa.eu/esco/occupation/81309031-dad2-4a7a-bde6-7f6e518f89ff': 
        'http://data.europa.eu/esco/occupation/f4525ed8-54eb-4a3b-90db-55cc01b0d9fd'
    }
    
    example[f'ESCO_uri_{i}'] = replacements_uri.get(example[f'ESCO_uri_{i}'], example[f'ESCO_uri_{i}'])
    
    return example

# Apply replacements to all columns in the dataset beginning with ESCO_title
for i in range(16):
    dataset['train'] = dataset['train'].map(lambda example: replace_esco_titles(example, i))
    dataset['validation'] = dataset['validation'].map(lambda example: replace_esco_titles(example, i))
    dataset['test'] = dataset['test'].map(lambda example: replace_esco_titles(example, i))


Create ESCO occupations dictionary.

In [8]:
# Load descriptions for ESCO occupations
ESCO_occupations = pd.read_csv(DATA_PATH / "occupations_en.csv")


# Create dictionary for ESCO occupations
ESCO_occupations_dict = ESCO_occupations.set_index("conceptUri")[
    "description"
].to_dict()

# Add to ESCO_occupations_dict keys which are the names of the occupations, and as value the description of the occupation
ESCO_occupations_dict.update(
    ESCO_occupations.set_index("preferredLabel")["description"].to_dict()
)

# For every occupation, go through the altLabels and add them to the dictionary
for index, row in ESCO_occupations.iterrows():
    # If there are no altLabels, skip
    if pd.isna(row["altLabels"]):
        continue
    for alt_label in row["altLabels"].split("\n"):
        ESCO_occupations_dict[alt_label] = row["description"]

# Per-occupation alt-labels lookup. Used to enrich the ESCO role text fed to
# SBERT during training: the preferred label alone is often terse, while alt
# labels add common synonyms (e.g. "head of technical", "technical supervisor"
# for "technical director") which make the target embedding more robust.
ESCO_uri_to_alt_labels = {}
ESCO_label_to_alt_labels = {}
for _, row in ESCO_occupations.iterrows():
    if pd.isna(row["altLabels"]):
        continue
    alts = [a.strip() for a in row["altLabels"].split("\n") if a and a.strip()]
    if not alts:
        continue
    joined = ", ".join(alts)
    ESCO_uri_to_alt_labels[row["conceptUri"]] = joined
    ESCO_label_to_alt_labels[row["preferredLabel"]] = joined
    for alt in alts:
        ESCO_label_to_alt_labels.setdefault(alt, joined)

# ISCO minor group (first 3 digits of `code`, e.g. "251") per ESCO
# occupation. There are 125 distinct prefixes in the data — we map each one
# to a stable integer id 0..124 and reserve id 125 for the unknown / padding
# sentinel. Used as a categorical per-step feature for the RNN.
_esco_prefixes = sorted({str(c)[:3] for c in ESCO_occupations["code"].dropna()})
ESCO_cat_to_id  = {p: i for i, p in enumerate(_esco_prefixes)}
NUM_ESCO_CATS = len(_esco_prefixes)   # 125 valid ids: 0..124
ESCO_CAT_UNK  = NUM_ESCO_CATS         # 125 = unknown / padding sentinel
ESCO_uri_to_cat = {
    row["conceptUri"]: ESCO_cat_to_id[str(row["code"])[:3]]
    for _, row in ESCO_occupations.iterrows()
}
print(f"ESCO category space: {NUM_ESCO_CATS} valid prefixes  (+ 1 unknown sentinel)")
print(f"occupations with alt labels: {len(ESCO_uri_to_alt_labels)}")

ESCO category space: 125 valid prefixes  (+ 1 unknown sentinel)
occupations with alt labels: 3011


Load a per-occupation feature block used as a **per-step input** to the RNN:

1. **Skills.** From `data/occupationSkillRelations_en.csv` — per-occupation simple mean SBERT embeddings of KNOWLEDGE labels (both essential and optional).

The skill block is concatenated to the LSTM cell input alongside the ESCO_experience and free-text role embeddings.

In [9]:
import pandas as pd
import torch.nn.functional as F

skills_df = pd.read_csv(DATA_PATH / "occupationSkillRelations_en.csv")
skills_df = skills_df.dropna(subset=["occupationUri", "skillLabel", "relationType", "skillType"])
# Keep only KNOWLEDGE entries (both essential/required and optional).
skills_df = skills_df[skills_df["skillType"] == "knowledge"]
print(f"skill relations (knowledge only): {len(skills_df)}  occupations: {skills_df['occupationUri'].nunique()}  "
      f"unique skill labels: {skills_df['skillLabel'].nunique()}")

# Encode every UNIQUE skill label once.
unique_skills = sorted(set(skills_df["skillLabel"].astype(str)))
print(f"encoding {len(unique_skills)} unique skill labels via SBERT...")
skill_emb_matrix = model.encode(
    [f"skill: {s}" for s in unique_skills],
    batch_size=256,
    convert_to_tensor=True,
    show_progress_bar=True,
)
skill_to_idx = {s: i for i, s in enumerate(unique_skills)}
skill_emb_dim = int(skill_emb_matrix.shape[1])

# Per-occupation SIMPLE mean SBERT embedding over knowledge entries
# (essential and optional pooled together with equal weight).
# Vectorized in a single index_add_ pass.
_df = skills_df[skills_df["skillLabel"].astype(str).isin(skill_to_idx)].copy()
_df["_sidx"] = _df["skillLabel"].astype(str).map(skill_to_idx).astype(int)

_uri_list   = _df["occupationUri"].drop_duplicates().tolist()
_uri_to_oid = {u: i for i, u in enumerate(_uri_list)}
_df["_oid"] = _df["occupationUri"].map(_uri_to_oid).astype(int)

_device = skill_emb_matrix.device
_oid_t  = torch.tensor(_df["_oid"].values,  dtype=torch.long, device=_device)
_sidx_t = torch.tensor(_df["_sidx"].values, dtype=torch.long, device=_device)

_sum_emb = torch.zeros(len(_uri_list), skill_emb_dim, device=_device)
_cnt     = torch.zeros(len(_uri_list), device=_device)
_sum_emb.index_add_(0, _oid_t, skill_emb_matrix[_sidx_t])
_cnt.index_add_(0, _oid_t, torch.ones_like(_oid_t, dtype=skill_emb_matrix.dtype))

_pooled_cpu = (_sum_emb / _cnt.unsqueeze(-1).clamp(min=1e-6)).cpu()
_cnt_cpu    = _cnt.cpu()
ESCO_uri_to_skills = {
    _uri_list[i]: _pooled_cpu[i]
    for i in range(len(_uri_list))
    if _cnt_cpu[i].item() > 0
}

print(f"occupations with skills: {len(ESCO_uri_to_skills)}  "
      f"(simple mean over knowledge entries, both essential and optional)")
print(f"skill embedding dim: {skill_emb_dim}")

ZERO_SKILL_EMB = torch.zeros(skill_emb_dim)

skill relations (knowledge only): 34384  occupations: 2965  unique skill labels: 3147
encoding 3147 unique skill labels via SBERT...


Batches: 100%|██████████| 13/13 [00:00<00:00, 18.16it/s]

occupations with skills: 2965  (simple mean over knowledge entries, both essential and optional)
skill embedding dim: 768


Find the latest start/end dates across all experiences per split, then collect each person's last experience to inspect the most recent ones per split.

In [10]:
def latest_date(split, field):                  
      best_key = None                             
      best_str = None                         
      for person in split:                        
          for i in range(person["number_of_experiences"]):         
              v = person[f"{field}_{i}"]  
              if v is None or v == "current":     
                  continue                        
              m, y = map(int, v.split("/"))
              key = (y, m)                        
              if best_key is None or key > best_key:
                  best_key = key                  
                  best_str = v
      return best_str                             
                  
                                              
for split_name, split in dataset.items():
      print(f"Split: {split_name}")
      print(f"Latest start: {latest_date(split,   
  'start')}")
      print(f"Latest end:   {latest_date(split,   
  'end')}")                                   
      print()                                  


def parse_date(s):                              
      if s is None or s == "current":           
          return None                         
      m, y = map(int, s.split("/"))       
      return (y, m)
                                                  
                                          
for split_name, split in dataset.items():       
      latest_experiences = []                     
      for person in split:                        
          last = person["number_of_experiences"] -   1                                              
          start = person[f"start_{last}"]         
          key = parse_date(start)             
          if key is None:                         
              continue
          latest_experiences.append((             
              key,
              person["identifier"],               
              start,                      
              person[f"end_{last}"],
          ))                                      
   
      latest_experiences.sort(key=lambda r: r[0]) 
                                          
      print(f"Split: {split_name}")
      for _, identifier, start, end in latest_experiences[:10]:
          print(f"  {identifier}  start={start}     end={end}")                                 
      print()  



Split: train
Latest start: 05/2021
Latest end:   08/2021

Split: validation
Latest start: 12/2020
Latest end:   05/2021

Split: test
Latest start: 10/2020
Latest end:   02/2021

Split: train
  94417768  start=04/1984     end=current
  14585273  start=01/1994     end=01/2008
  21629057  start=05/1994     end=05/2000
  30083943  start=07/1994     end=08/2015
  13411858  start=02/1995     end=current
  27689009  start=06/1995     end=current
  30083884  start=01/1996     end=current
  28243590  start=12/1996     end=current
  19147603  start=01/1997     end=04/2014
  30127072  start=01/1997     end=01/2002

Split: validation
  28398216  start=01/1997     end=04/2014
  24709432  start=04/2000     end=current
  26098594  start=01/2001     end=current
  26975573  start=01/2001     end=02/2011
  11813872  start=01/2003     end=current
  24592627  start=03/2004     end=09/2014
  33803142  start=01/2005     end=01/2015
  15553584  start=02/2005     end=05/2005
  36149549  start=11/2005     end=

Find earliest start and latest end.

In [11]:
def extreme_date_all_splits(dataset, field, mode):
    best_key = None
    best_str = None
    for split in dataset.values():
        for person in split:
            for i in range(person["number_of_experiences"]):
                v = person[f"{field}_{i}"]
                if v is None or v == "current":
                    continue
                m, y = map(int, v.split("/"))
                key = (y, m)
                if best_key is None or (key < best_key if mode == "earliest" else key > best_key):
                    best_key = key
                    best_str = v
    return best_str


print(f"Earliest start (all splits): {extreme_date_all_splits(dataset, 'start', 'earliest')}")
print(f"Latest end     (all splits): {extreme_date_all_splits(dataset, 'end', 'latest')}")


Earliest start (all splits): 02/1753
Latest end     (all splits): 08/2021


Inspect start dates earlier than 1950.

In [12]:
for split_name, split in dataset.items():
    for person in split:
        for i in range(person["number_of_experiences"]):
            v = person[f"start_{i}"]
            if v is None or v == "current":
                continue
            _, y = map(int, v.split("/"))
            if y < 1950:
                print(f"{split_name}  person_id={person['identifier']}  start={v}")


train  person_id=61677751  start=02/1753
train  person_id=61677751  start=02/1753
train  person_id=61677751  start=02/1753


Records only found for person with id 61677751. Remove it.

In [13]:
dataset = dataset.filter(lambda row: row["identifier"] != 61677751)


Filter out people with less than 2 experiences.  
Explode each remaining person's wide-format experiences into one row per experience.

In [14]:
from datasets import Dataset, DatasetDict

def explode_experiences(split):
    rows = []
    for person in split:
        for i in range(person["number_of_experiences"]):
            uri = person[f"ESCO_uri_{i}"]
            industry = person[f"industry_{i}"]
            if industry is None or (isinstance(industry, float) and industry != industry):
                industry = "<unk>"
            rows.append({
                "experience_id": person[f"uuid_{i}"],
                "person_id": person["identifier"],
                "experience_number": i,
                "title": person[f"title_{i}"],
                "description": person[f"description_{i}"],
                "summary": person[f"summary_{i}"],
                "industry": industry,
                "ESCO_uri": uri,
                "ESCO_title": person[f"ESCO_title_{i}"].strip(),
                "ESCO_cat": ESCO_uri_to_cat.get(uri, ESCO_CAT_UNK),
                "start": person[f"start_{i}"],
                "end": person[f"end_{i}"]
            })
    return Dataset.from_list(rows)

dataset = dataset.filter(lambda row:
  row["number_of_experiences"] >= 2)
dataset = DatasetDict({
    split: explode_experiences(dataset[split]) for split in dataset
})

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end'],
        num_rows: 7908
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end'],
        num_rows: 957
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end'],
        num_rows: 1050
    })
})


Replace "current" end dates with start+1 year (capped at 8/2021), then compute a months_of_experience field as the month difference between start and end for every experience row.

In [15]:
DATE_CAP = (2021, 8)

def months_between(start, end):
    sm, sy = map(int, start.split("/"))
    em, ey = map(int, end.split("/"))
    return (ey - sy) * 12 + (em - sm)

def fill_current_end(row):              
    if row["end"] != "current":                 
        return row                              
    m, y = map(int, row["start"].split("/"))    
    new_y, new_m = y + 1, m                     
    if (new_y, new_m) > DATE_CAP:                    
        new_y, new_m = DATE_CAP                      
    row["end"] = f"{new_m}/{new_y}"             
    return row
                                                  
                                              
dataset = dataset.map(fill_current_end)
dataset = dataset.map(lambda row: {"months_of_experience": months_between(row["start"], row["end"])})

Map: 100%|██████████| 1050/1050 [00:00<00:00, 20141.59 examples/s]


Inspect people with less than 0 months of experience.

In [16]:
for split_name, split in dataset.items():
    bad = [r for r in split if r["months_of_experience"] <= 0]
    print(f"{split_name}: {len(bad)} experiences with months_of_experience <= 0")
    for r in bad[:20]:
        print(f"  person_id={r['person_id']}  exp#={r['experience_number']}  "
              f"start={r['start']}  end={r['end']}  months={r['months_of_experience']}")


train: 83 experiences with months_of_experience <= 0
  person_id=47729453  exp#=1  start=01/2006  end=01/2006  months=0
  person_id=47729453  exp#=3  start=01/2007  end=01/2007  months=0
  person_id=47729453  exp#=5  start=01/2009  end=01/2009  months=0
  person_id=47729453  exp#=9  start=01/2016  end=01/2016  months=0
  person_id=18488289  exp#=1  start=06/2009  end=06/2009  months=0
  person_id=91318828  exp#=2  start=08/2014  end=01/2014  months=-7
  person_id=23497307  exp#=2  start=07/2017  end=03/2017  months=-4
  person_id=61319162  exp#=2  start=01/2008  end=01/2008  months=0
  person_id=22754014  exp#=0  start=01/2006  end=01/2006  months=0
  person_id=30642458  exp#=4  start=10/2013  end=10/2013  months=0
  person_id=45462344  exp#=7  start=08/2014  end=01/2011  months=-43
  person_id=10235429  exp#=6  start=01/2008  end=01/2008  months=0
  person_id=11522068  exp#=0  start=06/2011  end=06/2011  months=0
  person_id=11522068  exp#=1  start=06/2011  end=06/2011  months=0
  per

Remove people with less than 0 months of experience.

In [17]:
bad_persons = {
    split_name: {r["person_id"] for r in split if r["months_of_experience"] < 0}
    for split_name, split in dataset.items()
}
for split_name, ids in bad_persons.items():
    print(f"{split_name}: dropping {len(ids)} persons with negative-month experiences")

dataset = DatasetDict({
    split_name: split.filter(lambda r: r["person_id"] not in bad_persons[split_name])
    for split_name, split in dataset.items()
})
print(dataset)


train: dropping 17 persons with negative-month experiences
validation: dropping 5 persons with negative-month experiences
test: dropping 3 persons with negative-month experiences


Filter: 100%|██████████| 1050/1050 [00:00<00:00, 107258.14 examples/s]

DatasetDict({
    train: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end', 'months_of_experience'],
        num_rows: 7815
    })
    validation: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end', 'months_of_experience'],
        num_rows: 926
    })
    test: Dataset({
        features: ['experience_id', 'person_id', 'experience_number', 'title', 'description', 'summary', 'industry', 'ESCO_uri', 'ESCO_title', 'ESCO_cat', 'start', 'end', 'months_of_experience'],
        num_rows: 1039
    })
})


Transform free text experiences (history) and ESCO experience (target).

In [18]:
def free_text_experience(_experience_title, _experience_description):
    return f"role: {_experience_title} \n description: {_experience_description}"

def ESCO_experience(_ESCO_title, _ESCO_uri):
    description = ESCO_occupations_dict.get(_ESCO_uri)
    if description is None:
        description = ESCO_occupations_dict.get(_ESCO_title, "")
    alt_labels = (
        ESCO_uri_to_alt_labels.get(_ESCO_uri)
        or ESCO_label_to_alt_labels.get(_ESCO_title, "")
    )
    if alt_labels:
        return (
            f"esco role: {_ESCO_title} \n"
            f" alt labels: {alt_labels} \n"
            f" description: {description}"
        )
    return f"esco role: {_ESCO_title} \n description: {description}"

dataset = dataset.map(lambda row: {             
      "free_text_experience":                     
  free_text_experience(row["title"],          
  row["description"]),                            
      "ESCO_experience":
  ESCO_experience(row["ESCO_title"],              
  row["ESCO_uri"]),                       
  })

Map: 100%|██████████| 1039/1039 [00:00<00:00, 11368.72 examples/s]


Group experiences by person, sort them chronologically, then for each person enumerate every contiguous subspan of length ≥2 as a (prefix history, prefix months, prefix ESCO embeddings, **prefix ESCO URIs**, last-prefix summary, target ESCO) tuple — keeping only the last 16 per person. Per-step `ESCO_uri` is added as a key for the **per-step skills lookup** (simple mean SBERT skill embedding over knowledge entries, both essential and optional) at encode time. The summary is a single sequence-level feature drawn from the most recent role in the prefix (index `j + L - 2`), NOT from the target.

In [19]:
from collections import defaultdict


def build_pairs(split):
    by_person = defaultdict(list)
    for row in split:
        by_person[row["person_id"]].append(row)

    pairs = []
    for exps in by_person.values():
        exps.sort(key=lambda r: r["experience_number"])
        n = len(exps)
        person_pairs = []
        for L in range(2, n + 1):           # subspan length
            for j in range(n - L + 1):      # start position
                prefix        = [r["free_text_experience"] for r in exps[j : j + L - 1]]
                prefix_months = [r["months_of_experience"] for r in exps[j : j + L - 1]]
                # Per-step ESCO URI — used at encode time to look up per-occupation
                # mean SBERT embeddings over knowledge entries (both essential and optional).
                prefix_uris   = [r["ESCO_uri"]             for r in exps[j : j + L - 1]]
                # Sequence-level summary: only the LAST prefix experience's summary.
                last_summary = exps[j + L - 2]["summary"]
                if last_summary is None or (isinstance(last_summary, float) and last_summary != last_summary):
                    last_summary = ""
                target = exps[j + L - 1]["ESCO_experience"]
                person_pairs.append((prefix, prefix_months,
                                     prefix_uris, last_summary, target))
        pairs.extend(person_pairs[-16:])

    return pairs


pairs = {split: build_pairs(dataset[split]) for split in dataset}

for split, p in pairs.items():
    print(f"{split}: {len(p)} pairs")


train: 13311 pairs
validation: 1489 pairs
test: 1787 pairs


Encode every prefix experience text and target ESCO text into embeddings, encode the per-sequence summary into a single embedding, build a log1p(months) duration scalar per step, and the **per-user skill vector** (mean SBERT skill embedding of the LAST prefix role's ESCO_uri — i.e. the user's second-to-last role, not the target).


In [20]:
import torch

def encode_pairs(pairs):
    (prefix_texts_list, prefix_months_list,
     prefix_uris_list,
     summary_texts, esco_texts) = zip(*pairs)

    print("Embedding career history, summaries, and ESCO targets...")

    # Encode each free-text experience once — one step per role.
    lengths = [len(seq) for seq in prefix_texts_list]
    flat_texts = [t for seq in prefix_texts_list for t in seq]
    flat_embeddings = model.encode(
        flat_texts,
        batch_size=256,
        convert_to_tensor=True,
        show_progress_bar=True,
    )
    per_pair_embeddings = list(torch.split(flat_embeddings, lengths))

    # Per-step duration feature.
    per_pair_months = [
        torch.log1p(torch.tensor(m, dtype=torch.float32, device=flat_embeddings.device))
        for m in prefix_months_list
    ]

    # Per-USER skill embedding — simple mean SBERT over knowledge entries (both essential and optional) of
    # the LAST prefix role's ESCO_uri (the user's second-to-last role, NOT the
    # target). Unknown URIs fall back to a zero vector.
    skill_rows = []
    for uris in prefix_uris_list:
        last_uri = uris[-1]
        skill_rows.append(ESCO_uri_to_skills.get(last_uri, ZERO_SKILL_EMB))
    per_pair_skills = torch.stack(skill_rows, dim=0)                           # [N, D]

    # Per-pair summary embedding.
    summary_embeddings = model.encode(
        list(summary_texts),
        batch_size=256,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    # Target ESCO embedding (next role).
    esco_occupation_embeddings = model.encode(
        list(esco_texts),
        batch_size=256,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    return (per_pair_embeddings, per_pair_months,
            per_pair_skills,
            summary_embeddings, esco_occupation_embeddings)


(train_history_embeddings, train_history_months,
 train_history_skills,
 train_summary_embeddings, train_esco_embeddings) = encode_pairs(pairs["train"])
(val_history_embeddings, val_history_months,
 val_history_skills,
 val_summary_embeddings, val_esco_embeddings) = encode_pairs(pairs["validation"])

embedding_dim = train_esco_embeddings.shape[1]
skills_dim_in = train_history_skills.shape[-1]                                 # = skill_emb_dim
train_lens = [t.size(0) for t in train_history_embeddings]
print(f"Train: {len(train_history_embeddings)} sequences, embedding_dim={embedding_dim}")
print(f"  prefix lens - min={min(train_lens)}  max={max(train_lens)}  mean={sum(train_lens)/len(train_lens):.1f}")
print(f"  summary embeddings:           {tuple(train_summary_embeddings.shape)}")
print(f"  per-user skills:              {tuple(train_history_skills.shape)}  (input dim={skills_dim_in})")
print(f"Val:   {len(val_history_embeddings)} sequences")


Embedding career history, summaries, and ESCO targets...


Batches: 100%|██████████| 52/52 [00:27<00:00,  1.91it/s]


Embedding career history, summaries, and ESCO targets...


Batches: 100%|██████████| 6/6 [00:03<00:00,  1.63it/s]

Train: 13311 sequences, embedding_dim=768
  prefix lens - min=1  max=16  mean=2.6
  summary embeddings:           (13311, 768)
  per-user skills:              (13311, 768)  (input dim=768)
Val:   1489 sequences


Wrap the per-pair history embeddings, month scalars, **per-user skill vectors** (one per pair, taken from the last prefix role), summary embeddings, and ESCO targets into a `CareerHistoryDataset`, and build train/val DataLoaders with a custom collate that pads variable-length sequences.


In [21]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


class CareerHistoryDataset(Dataset):
    def __init__(self, history_embeddings, history_months,
                 history_skills,
                 summary_embeddings, esco_embeddings):
        n = len(history_embeddings)
        assert (n == len(esco_embeddings) == len(history_months)
                == len(history_skills)
                == len(summary_embeddings))
        self.history       = history_embeddings
        self.months        = history_months
        self.skills        = history_skills
        self.summary       = summary_embeddings
        self.esco          = esco_embeddings

    def __len__(self):
        return len(self.history)

    def __getitem__(self, idx):
        return (self.history[idx], self.months[idx],
                self.skills[idx],
                self.summary[idx], self.esco[idx])


def collate_pad(batch):
    seqs, months, skills, summaries, targets = zip(*batch)
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    padded         = pad_sequence(seqs,       batch_first=True)                                     # [B, L, D]
    padded_months  = pad_sequence(months,     batch_first=True)                                     # [B, L]
    skills_b       = torch.stack(list(skills),    dim=0)                                            # [B, D_skill]
    summaries      = torch.stack(list(summaries), dim=0)                                            # [B, D]
    targets        = torch.stack(list(targets),   dim=0)                                            # [B, D]
    return (padded, padded_months, skills_b,
            summaries, lengths, targets)


batch_size = config["neural"]["batch_size"]

train_dataset = CareerHistoryDataset(
    train_history_embeddings, train_history_months,
    train_history_skills,
    train_summary_embeddings, train_esco_embeddings
)
val_dataset = CareerHistoryDataset(
    val_history_embeddings, val_history_months,
    val_history_skills,
    val_summary_embeddings, val_esco_embeddings
)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,  collate_fn=collate_pad
)
val_loader = DataLoader(
    val_dataset,   batch_size=batch_size, shuffle=False, collate_fn=collate_pad
)

(padded, padded_months, batch_skills,
 summaries, lengths, targets) = next(iter(train_loader))
print(f"padded:        {tuple(padded.shape)}")
print(f"padded_months: {tuple(padded_months.shape)}")
print(f"batch_skills:  {tuple(batch_skills.shape)}")
print(f"summaries:     {tuple(summaries.shape)}")
print(f"lengths:       {lengths.tolist()}")
print(f"targets:       {tuple(targets.shape)}")


padded:        (256, 10, 768)
padded_months: (256, 10)
batch_skills:  (256, 768)
summaries:     (256, 768)
lengths:       [1, 5, 1, 1, 2, 2, 4, 2, 1, 1, 2, 4, 3, 9, 2, 4, 6, 3, 4, 3, 3, 1, 1, 2, 4, 8, 2, 1, 2, 1, 2, 1, 8, 3, 7, 1, 1, 1, 1, 4, 5, 1, 3, 4, 2, 2, 3, 4, 1, 1, 1, 2, 6, 5, 1, 2, 1, 5, 1, 9, 3, 7, 10, 3, 3, 1, 3, 2, 4, 2, 1, 4, 5, 1, 1, 1, 1, 2, 1, 3, 4, 6, 5, 1, 1, 2, 2, 2, 4, 1, 3, 1, 3, 1, 2, 2, 4, 1, 5, 2, 2, 3, 5, 2, 3, 1, 4, 3, 4, 3, 3, 1, 2, 4, 3, 5, 1, 1, 6, 1, 3, 1, 3, 2, 4, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 3, 1, 2, 1, 7, 2, 2, 2, 3, 3, 1, 2, 2, 1, 1, 1, 3, 4, 9, 2, 4, 2, 2, 2, 2, 2, 3, 2, 1, 2, 4, 2, 3, 3, 1, 7, 1, 1, 4, 2, 3, 1, 1, 5, 4, 1, 1, 3, 4, 4, 5, 1, 1, 1, 1, 2, 2, 3, 2, 1, 4, 3, 3, 2, 1, 7, 1, 3, 1, 3, 2, 2, 4, 2, 1, 2, 1, 6, 4, 2, 2, 1, 1, 9, 5, 1, 2, 1, 1, 4, 1, 2, 8, 8, 1, 4, 3, 5, 2, 3, 2, 1, 1, 2, 3, 4, 6, 1, 1, 2, 1, 1, 2, 4, 2, 1, 2, 1]
targets:       (256, 768)


Train a custom single-layer LSTM on the per-role history embeddings. **Per-step input** combines two signals:
- free-text role embedding (`model.encode` of `"role: title / description: …"`),
- learned projection of `log1p(months_of_experience)`.

Sequence-level summary is projected (Tanh-bounded) to `hidden_dim` and used as the LSTM's initial hidden state `h0`; cell state `c0` is initialized to zeros. The hidden state at the last valid timestep is taken as the sequence representation (no attention pooling). A **per-user skill vector** (mean SBERT skill embedding of the LAST prefix role's ESCO_uri) is projected and concatenated to the pooled LSTM output before the final linear projection — i.e. skills inform the prediction directly, without flowing through the recurrent dynamics. Training uses InfoNCE with in-batch positives + an external label bank of every distinct ESCO_experience target text.

## Reproducibility & regenerate-from-scratch controls

Run this **before** training. It frees the GPU cache and seeds the run.

The **final** result is the 10-model ensemble below, which seeds itself `0..9` (`_set_seed`) — so it stays **reproducible** regardless of this cell. The scoring cell reuses the on-disk ensemble checkpoints (`output/lstm-noindustry_seed*.pth`) whenever they exist, so the final is stable. To rebuild it from scratch, set `REGENERATE_FROM_SCRATCH = True`, run this cell, then re-run the **ensemble** cell and the **scoring** cell.

The single-model cell right below has no seed of its own, so it follows `RUN_SEED` here, which is drawn **randomly** — each single-model run differs (the seed is printed, so you can reproduce a given run by pinning `RUN_SEED`).

In [ ]:
# === Reproducibility & "regenerate from scratch" controls ====================
# Run this cell before training.
#
# The FINAL result is the 10-model ENSEMBLE below, which seeds itself 0..9
# (_set_seed) -> it is reproducible and does NOT depend on this cell. The
# scoring cell reuses the on-disk ensemble checkpoints whenever they exist, so
# the final stays stable. To rebuild it from scratch, set
# REGENERATE_FROM_SCRATCH = True, run this cell, then re-run the ensemble cell
# and the scoring cell.
#
# The SINGLE-model cell right below has no seed of its own, so it follows the
# RUN_SEED set here -> we draw it RANDOMLY, so each single-model run differs.
# The seed is printed, so you can still reproduce a given run by pinning it.
import gc, os, glob, random
import numpy as np

REGENERATE_FROM_SCRATCH = True   # True -> wipe this model's checkpoints/outputs and rebuild

# --- Random seed for the single-model run (the ensemble uses its own 0..9) ---
RUN_SEED = int.from_bytes(os.urandom(4), "little")   # pin to an int to reproduce a run
random.seed(RUN_SEED)
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RUN_SEED)
torch.backends.cudnn.deterministic = True            # keeps a *pinned* seed reproducible
torch.backends.cudnn.benchmark = False

# --- Free GPU memory held by any previous run --------------------------------
for _name in ("rnn_model", "optimizer", "m", "opt"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# --- Optional: wipe artifacts for a true from-scratch rebuild ----------------
# .pth files are gitignored (rebuilt by retraining); the .json/.pkl are git-
# tracked (recover with `git checkout -- <file>` if needed).
if REGENERATE_FROM_SCRATCH:
    _targets = (glob.glob("./output/lstm-noindustry_seed*.pth")
                + ["./output/lstm-noindustry.pth",
                   "./output/decorte_scores_lstm-noindustry.json",
                   "./output/decorte_predictions_lstm-noindustry.pkl"])
    removed = 0
    for _p in _targets:
        if os.path.exists(_p):
            os.remove(_p); removed += 1
            print("removed", _p)
    print(f"cleared {removed} artifact(s) -> now re-run the ENSEMBLE cell, then the SCORING cell")

_mem = (f"  GPU allocated={torch.cuda.memory_allocated() / 1e6:.1f} MB"
        if torch.cuda.is_available() else "")
print(f"RUN_SEED={RUN_SEED} (single-model, random); ensemble stays fixed at seeds 0..9.{_mem}")
# =============================================================================

In [ ]:
# Model architecture hyperparameters — defined once here and reused by
# the single-model, ensemble, and test/prediction builds below.
MONTHS_DIM = 16
SKILL_DIM = 64
BIDIRECTIONAL = True

In [22]:
import os
import torch.nn as nn
import torch.nn.functional as F

RNN_CKPT_PATH = "./output/lstm-noindustry.pth"
os.makedirs("output", exist_ok=True)


class CustomLSTMCell(nn.Module):
    """
    LSTM cell that concatenates per-step months to the free-text role
    embedding before applying the gates.
    """
    def __init__(self, input_dim, hidden_dim, months_dim):
        super().__init__()
        gate_in = input_dim + months_dim
        self.x2h = nn.Linear(gate_in, 4 * hidden_dim)
        self.h2h = nn.Linear(hidden_dim, 4 * hidden_dim)

    def forward(self, x, m, h, cell):
        combined = torch.cat([x, m], dim=-1)
        gates = self.x2h(combined) + self.h2h(h)
        i_g, f_g, g_g, o_g = gates.chunk(4, dim=-1)
        i_g = torch.sigmoid(i_g)
        f_g = torch.sigmoid(f_g)
        g_g = torch.tanh(g_g)
        o_g = torch.sigmoid(o_g)
        new_cell = f_g * cell + i_g * g_g
        new_h = o_g * torch.tanh(new_cell)
        return new_h, new_cell


class CustomLSTM(nn.Module):
    """
    Custom single-layer LSTM that returns the hidden state at the last valid timestep.
    Per-step input is the concatenation of free-text role and months.
    Sequence-level summary is projected (Tanh) to hidden_dim and used as h0;
    cell state c0 is initialized to zeros.
    Skills are NOT consumed by the cell — they are concatenated to the
    pooled output by `CareerRNN`. When `bidirectional=True`, a second LSTM
    cell runs over the reversed sequence (length-aware so padding doesn't
    pollute the backward initialization). The final-step forward hidden state is concatenated with the final-step backward hidden state when bidirectional.
    """
    def __init__(self, input_dim, hidden_dim, months_dim,
                 dropout=0.0,
                 bidirectional=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.bidirectional = bidirectional
        self.months_proj = nn.Sequential(
            nn.Linear(1, months_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.summary_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
        )
        self.cell_fwd = CustomLSTMCell(input_dim, hidden_dim, months_dim)
        if bidirectional:
            self.cell_bwd = CustomLSTMCell(input_dim, hidden_dim, months_dim)

    @property
    def output_dim(self):
        return self.hidden_dim * (2 if self.bidirectional else 1)

    def forward(self, padded, months, summary, lengths):
        B, L, _ = padded.shape
        m_all = self.months_proj(months.unsqueeze(-1))                    # [B, L, months_dim]
        h0 = self.summary_proj(summary)                                   # [B, hidden_dim]
        c0 = torch.zeros_like(h0)                                         # [B, hidden_dim]

        # Forward direction.
        h, cell = h0, c0
        outputs_fwd = []
        for t in range(L):
            h, cell = self.cell_fwd(padded[:, t, :], m_all[:, t, :], h, cell)
            outputs_fwd.append(h)
        outputs_fwd = torch.stack(outputs_fwd, dim=1)                     # [B, L, H]

        lengths_dev = lengths.to(outputs_fwd.device)
        last_idx = (lengths_dev - 1).clamp(min=0)
        last_fwd = outputs_fwd[torch.arange(B, device=outputs_fwd.device), last_idx]

        if self.bidirectional:
            # Backward direction with length-aware masking: a sequence of
            # length L_i has its first backward step at position L_i-1, so
            # for t >= L_i we keep the (h, cell) at their initial values
            # (i.e. don't let padded positions corrupt the backward init).
            h_b, cell_b = h0, c0
            for t in range(L - 1, -1, -1):
                step_mask = (lengths_dev > t).float().unsqueeze(-1)       # [B, 1]
                h_new, cell_new = self.cell_bwd(padded[:, t, :], m_all[:, t, :], h_b, cell_b)
                h_b    = step_mask * h_new    + (1.0 - step_mask) * h_b
                cell_b = step_mask * cell_new + (1.0 - step_mask) * cell_b
            return torch.cat([last_fwd, h_b], dim=-1)

        return last_fwd


class CareerRNN(nn.Module):
    """
    v3 career-trajectory LSTM. Per-step features = free-text role and months.
    **Skills** (simple mean SBERT skill embedding over knowledge entries (both essential and optional) of the LAST prefix
    role's ESCO_uri — the user's second-to-last profession) are NOT fed into
    the LSTM cell; they are projected and concatenated to the pooled LSTM
    output before the final linear projection.

    Input:
      padded   [B, L, D_in]
      months   [B, L]
      skills   [B, D_skill]
      summary  [B, D_in]
      lengths  [B]
    Output:
      [B, D_out]
    """
    def __init__(self, input_dim, hidden_dim, output_dim,
                 dropout=0.0, months_dim=16,
                 skill_input_dim=None, skill_dim=64,
                 bidirectional=True):
        super().__init__()
        if skill_input_dim is None:
            skill_input_dim = 2 * input_dim
        self.lstm = CustomLSTM(input_dim, hidden_dim, months_dim,
                               dropout,
                               bidirectional=bidirectional)
        # Per-user skills projection: mean SBERT skill vector -> skill_dim.
        self.skills_proj = nn.Sequential(
            nn.Linear(skill_input_dim, skill_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
        )
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(self.lstm.output_dim + skill_dim, output_dim)

    def forward(self, padded, months, skills, summary, lengths):
        pooled = self.lstm(padded, months, summary, lengths)              # [B, output_dim_lstm]
        s = self.skills_proj(skills)                                      # [B, skill_dim]
        return self.proj(self.drop(torch.cat([pooled, s], dim=-1)))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rnn_model = CareerRNN(
    input_dim=embedding_dim,
    hidden_dim=config["neural"]["hidden_sizes"][0],
    output_dim=embedding_dim,
    dropout=config["neural"]["dropout_rate"] if config["neural"]["dropout"] else 0.0,
    months_dim=MONTHS_DIM,
    skill_input_dim=skills_dim_in,
    skill_dim=SKILL_DIM,
    bidirectional=BIDIRECTIONAL,
).to(device)

optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=1e-4, weight_decay=1e-2)


# Full ESCO label bank — for training negatives and val MRR.
_all_esco_texts = sorted({target for sp in pairs.values() for *_, target in sp})
_text_to_label_id = {t: i for i, t in enumerate(_all_esco_texts)}
_label_bank = model.encode(
    _all_esco_texts, batch_size=256, convert_to_tensor=True, show_progress_bar=False
)
_label_bank = F.normalize(_label_bank, dim=-1).to(device)


def info_nce_with_bank(pred, target, bank, tau=0.05, n_bank_neg=4096):
    B = target.size(0)
    p = F.normalize(pred, dim=-1)
    t = F.normalize(target, dim=-1)

    idx = torch.randint(0, bank.size(0), (n_bank_neg,), device=bank.device)
    bank_neg = bank[idx]

    keys = torch.cat([t, bank_neg], dim=0)
    logits = (p @ keys.T) / tau

    with torch.no_grad():
        tt_sim = t @ t.T
        eye = torch.eye(B, dtype=torch.bool, device=t.device)
        in_batch_dup = (tt_sim > 0.999) & ~eye
        bank_sim = t @ bank_neg.T
        bank_dup = bank_sim > 0.999
        dup_mask = torch.cat([in_batch_dup, bank_dup], dim=1)
    logits = logits.masked_fill(dup_mask, float("-inf"))

    labels = torch.arange(B, device=p.device)
    return F.cross_entropy(logits, labels)


@torch.no_grad()
def eval_mrr(loader):
    rnn_model.eval()
    all_preds, all_targets = [], []
    for (padded, padded_months, batch_skills,
         summaries, lengths, targets) in loader:
        padded        = padded.to(device)
        padded_months = padded_months.to(device)
        batch_skills  = batch_skills.to(device)
        summaries     = summaries.to(device)
        preds = rnn_model(padded, padded_months,
                          batch_skills, summaries, lengths)
        all_preds.append(F.normalize(preds, dim=-1))
        all_targets.append(F.normalize(targets.to(device), dim=-1))
    all_preds   = torch.cat(all_preds, 0)
    all_targets = torch.cat(all_targets, 0)

    sim_to_bank = all_targets @ _label_bank.T
    true_ids = sim_to_bank.argmax(dim=1)

    scores = all_preds @ _label_bank.T
    true_scores = scores.gather(1, true_ids.unsqueeze(1))
    ranks = (scores > true_scores).sum(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()


@torch.no_grad()
def eval_val_mrr(loader, pair_list):
    rnn_model.eval()
    all_preds = []
    for (padded, padded_months, batch_skills,
         summaries, lengths, _) in loader:
        padded        = padded.to(device)
        padded_months = padded_months.to(device)
        batch_skills  = batch_skills.to(device)
        summaries     = summaries.to(device)
        preds  = rnn_model(padded, padded_months,
                           batch_skills, summaries, lengths)
        all_preds.append(F.normalize(preds, dim=-1))
    all_preds = torch.cat(all_preds, dim=0)

    true_ids = torch.tensor(
        [_text_to_label_id[t] for *_, t in pair_list], device=device
    )
    scores = all_preds @ _label_bank.T
    true_scores = scores.gather(1, true_ids.unsqueeze(1))
    ranks = (scores > true_scores).sum(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()


def train_epoch(loader):
    rnn_model.train()
    total, n = 0.0, 0
    for (padded, padded_months, batch_skills,
         summaries, lengths, targets) in loader:
        padded        = padded.to(device)
        padded_months = padded_months.to(device)
        batch_skills  = batch_skills.to(device)
        summaries     = summaries.to(device)
        targets       = targets.to(device)
        preds = rnn_model(padded, padded_months,
                          batch_skills, summaries, lengths)
        loss  = info_nce_with_bank(preds, targets, _label_bank)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(rnn_model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item() * targets.size(0)
        n     += targets.size(0)
    return total / n


patience_cfg = max(config["neural"]["patience"], 3)
best_mrr, best_epoch, patience_left = 0.0, 0, patience_cfg
for epoch in range(1, config["neural"]["epochs"] + 1):
    train_loss = train_epoch(train_loader)
    train_mrr = eval_mrr(train_loader)
    val_mrr    = eval_val_mrr(val_loader, pairs["validation"])
    print(f"epoch {epoch:>2}  train_loss={train_loss:.4f}  train_MRR={train_mrr:.4f}  val_MRR={val_mrr:.4f}")

    if val_mrr > best_mrr:
        best_mrr, best_epoch, patience_left = val_mrr, epoch, patience_cfg
        torch.save(rnn_model.state_dict(), RNN_CKPT_PATH)
    else:
        patience_left -= 1
        if patience_left == 0:
            print("early stop")
            break

print(f"best val MRR: {best_mrr:.4f} at epoch {best_epoch}  checkpoint: {RNN_CKPT_PATH}")


epoch  1  train_loss=7.5208  train_MRR=0.2198  val_MRR=0.2159
epoch  2  train_loss=6.3770  train_MRR=0.2939  val_MRR=0.2620
epoch  3  train_loss=5.9676  train_MRR=0.3233  val_MRR=0.2759
epoch  4  train_loss=5.7551  train_MRR=0.3442  val_MRR=0.2851
epoch  5  train_loss=5.6096  train_MRR=0.3630  val_MRR=0.2883
epoch  6  train_loss=5.4805  train_MRR=0.3762  val_MRR=0.2876
epoch  7  train_loss=5.3749  train_MRR=0.3887  val_MRR=0.2906
epoch  8  train_loss=5.2931  train_MRR=0.4003  val_MRR=0.2912
epoch  9  train_loss=5.2117  train_MRR=0.4117  val_MRR=0.2898
epoch 10  train_loss=5.1229  train_MRR=0.4208  val_MRR=0.2899
epoch 11  train_loss=5.0557  train_MRR=0.4294  val_MRR=0.2928
epoch 12  train_loss=4.9896  train_MRR=0.4405  val_MRR=0.2903
epoch 13  train_loss=4.9120  train_MRR=0.4491  val_MRR=0.2890
epoch 14  train_loss=4.8487  train_MRR=0.4588  val_MRR=0.2912
epoch 15  train_loss=4.7914  train_MRR=0.4680  val_MRR=0.2869
epoch 16  train_loss=4.7223  train_MRR=0.4776  val_MRR=0.2839
epoch 17

## Multi-seed ensemble

Train K independent copies of the same LSTM model from different seeds and average their **prediction embeddings** before FAISS search. Per-seed val MRR varies by ±0.005 due to optimizer noise (init / batch shuffle / dropout masks); averaging cancels per-seed val-fitting and typically yields +0.5–1 p.p. test MRR over a single seed.

Per-seed checkpoints are saved to `./output/lstm-noindustry_seed{i}.pth` and the testing cell automatically uses them if present (falls back to the single `RNN_CKPT_PATH` otherwise).

In [23]:
import math
import os
import random

K_SEEDS = 10
ENSEMBLE_RANDOM_SEEDS = False   # False -> fixed seeds 0..K_SEEDS-1 (reproducible); True -> random
ensemble_seeds = ([int.from_bytes(os.urandom(4), "little") for _ in range(K_SEEDS)]
                  if ENSEMBLE_RANDOM_SEEDS else list(range(K_SEEDS)))
print(f"ensemble seeds: {ensemble_seeds}")
ENSEMBLE_CKPT_PATHS = [f"./output/lstm-noindustry_seed{i}.pth" for i in range(K_SEEDS)]


def _set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_gru_net():
    return CareerRNN(
        input_dim=embedding_dim,
        hidden_dim=config["neural"]["hidden_sizes"][0],
        output_dim=embedding_dim,
        dropout=config["neural"]["dropout_rate"] if config["neural"]["dropout"] else 0.0,
        months_dim=MONTHS_DIM,
        skill_input_dim=skills_dim_in,
        skill_dim=SKILL_DIM,
    bidirectional=BIDIRECTIONAL,
).to(device)


def _train_seed(seed, ckpt_path):
    _set_seed(seed)
    m = _build_gru_net()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-2)

    @torch.no_grad()
    def _val(model):
        model.eval()
        all_p = []
        for (padded, padded_months, batch_skills,
             summaries, lengths, _t) in val_loader:
            padded, padded_months = padded.to(device), padded_months.to(device)
            batch_skills = batch_skills.to(device)
            summaries = summaries.to(device)
            preds = model(padded, padded_months,
                          batch_skills, summaries, lengths)
            all_p.append(F.normalize(preds, dim=-1))
        all_p = torch.cat(all_p, dim=0)
        true_ids = torch.tensor([_text_to_label_id[t] for *_u, t in pairs["validation"]], device=device)
        scores = all_p @ _label_bank.T
        ts = scores.gather(1, true_ids.unsqueeze(1))
        ranks = (scores > ts).sum(dim=1) + 1
        return (1.0 / ranks.float()).mean().item()

    pat_cfg = max(config["neural"]["patience"], 3)
    best_v, best_e, pat = 0.0, 0, pat_cfg
    for ep in range(1, config["neural"]["epochs"] + 1):
        m.train()
        total, n = 0.0, 0
        for (padded, padded_months, batch_skills,
             summaries, lengths, targets) in train_loader:
            padded, padded_months = padded.to(device), padded_months.to(device)
            batch_skills = batch_skills.to(device)
            summaries, targets = summaries.to(device), targets.to(device)
            preds = m(padded, padded_months,
                      batch_skills, summaries, lengths)
            loss = info_nce_with_bank(preds, targets, _label_bank)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            opt.step()
            total += loss.item() * targets.size(0)
            n += targets.size(0)
        v = _val(m)
        print(f"  [seed {seed}] ep {ep:>2}  loss={total/n:.4f}  val_MRR={v:.4f}")
        if v > best_v:
            best_v, best_e, pat = v, ep, pat_cfg
            torch.save(m.state_dict(), ckpt_path)
        else:
            pat -= 1
            if pat == 0:
                print(f"  [seed {seed}] early stop at epoch {ep}")
                break
    print(f"  [seed {seed}] best val MRR: {best_v:.4f} at epoch {best_e}  ckpt: {ckpt_path}")
    return best_v


seed_results = []
for i in range(K_SEEDS):
    print(f"\n=== Training member {i} (seed {ensemble_seeds[i]}) ===")
    seed_results.append(_train_seed(ensemble_seeds[i], ENSEMBLE_CKPT_PATHS[i]))


# Ensemble val MRR — average normalized predictions across all K models.
@torch.no_grad()
def ensemble_val_mrr(ckpt_paths, loader, pair_list):
    sum_preds = None
    for path in ckpt_paths:
        m = _build_gru_net()
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        chunk = []
        for (padded, padded_months, batch_skills,
             summaries, lengths, _t) in loader:
            padded, padded_months = padded.to(device), padded_months.to(device)
            batch_skills = batch_skills.to(device)
            summaries = summaries.to(device)
            preds = m(padded, padded_months,
                      batch_skills, summaries, lengths)
            chunk.append(F.normalize(preds, dim=-1))
        chunk = torch.cat(chunk, dim=0)
        sum_preds = chunk if sum_preds is None else sum_preds + chunk
    avg = F.normalize(sum_preds / len(ckpt_paths), dim=-1)
    true_ids = torch.tensor([_text_to_label_id[t] for *_u, t in pair_list], device=device)
    scores = avg @ _label_bank.T
    ts = scores.gather(1, true_ids.unsqueeze(1))
    ranks = (scores > ts).sum(dim=1) + 1
    return (1.0 / ranks.float()).mean().item()


print()
print("=== Per-seed val MRR ===")
for i, v in enumerate(seed_results):
    print(f"  seed {i}: {v:.4f}")
print(f"  mean of best-vals: {sum(seed_results)/len(seed_results):.4f}")
ens_v = ensemble_val_mrr(ENSEMBLE_CKPT_PATHS, val_loader, pairs["validation"])
print(f"  ENSEMBLE (averaged predictions): {ens_v:.4f}")
print(f"  reference single-model (RNN_CKPT_PATH): see best val MRR printed above")



=== Training seed 0 ===
  [seed 0] ep  1  loss=7.5192  val_MRR=0.2247
  [seed 0] ep  2  loss=6.3671  val_MRR=0.2659
  [seed 0] ep  3  loss=5.9495  val_MRR=0.2793
  [seed 0] ep  4  loss=5.7360  val_MRR=0.2830
  [seed 0] ep  5  loss=5.5919  val_MRR=0.2867
  [seed 0] ep  6  loss=5.4738  val_MRR=0.2920
  [seed 0] ep  7  loss=5.3791  val_MRR=0.2888
  [seed 0] ep  8  loss=5.2803  val_MRR=0.2916
  [seed 0] ep  9  loss=5.2016  val_MRR=0.2910
  [seed 0] ep 10  loss=5.1267  val_MRR=0.2926
  [seed 0] ep 11  loss=5.0564  val_MRR=0.2938
  [seed 0] ep 12  loss=4.9726  val_MRR=0.2978
  [seed 0] ep 13  loss=4.9107  val_MRR=0.2929
  [seed 0] ep 14  loss=4.8421  val_MRR=0.2919
  [seed 0] ep 15  loss=4.7831  val_MRR=0.2925
  [seed 0] ep 16  loss=4.7167  val_MRR=0.2976
  [seed 0] ep 17  loss=4.6468  val_MRR=0.2940
  [seed 0] ep 18  loss=4.5834  val_MRR=0.2934
  [seed 0] early stop at epoch 18
  [seed 0] best val MRR: 0.2978 at epoch 12  ckpt: ./output/gru_model_decorte_v2_1_seed0.pth

=== Training seed 1

## Testing the model

In [24]:
import json
import pickle
import os
import numpy as np
import faiss

SCORES_PATH      = "./output/decorte_scores_lstm-noindustry.json"
PREDICTIONS_PATH = "./output/decorte_predictions_lstm-noindustry.pkl"

# 1. Encode test pairs.
(test_history_embeddings, test_history_months,
 test_history_skills,
 test_summary_embeddings, test_esco_embeddings) = encode_pairs(pairs["test"])

# 2. Build ESCO label space.
all_esco_texts = sorted({
    target
    for split_pairs in pairs.values()
    for *_, target in split_pairs
})
label_embeddings = model.encode(
    all_esco_texts, batch_size=256, convert_to_tensor=True, show_progress_bar=True
)
label_embeddings = F.normalize(label_embeddings, dim=-1)
label_embeddings_np = label_embeddings.cpu().numpy().astype("float32")

index = faiss.IndexFlatIP(label_embeddings_np.shape[1])
index.add(label_embeddings_np)

text_to_label_id = {t: i for i, t in enumerate(all_esco_texts)}

# 3. Decide which checkpoints to use: ensemble if present, else single.
try:
    _ENS_PATHS = ENSEMBLE_CKPT_PATHS
except NameError:
    _ENS_PATHS = []
ckpt_paths = [p for p in _ENS_PATHS if os.path.exists(p)]
if len(ckpt_paths) >= 2:
    print(f"using ENSEMBLE of {len(ckpt_paths)} checkpoints")
else:
    ckpt_paths = [RNN_CKPT_PATH]
    print(f"using SINGLE checkpoint: {RNN_CKPT_PATH}")


def _build_test_model():
    return CareerRNN(
        input_dim=embedding_dim,
        hidden_dim=config["neural"]["hidden_sizes"][0],
        output_dim=embedding_dim,
        dropout=config["neural"]["dropout_rate"] if config["neural"]["dropout"] else 0.0,
        months_dim=MONTHS_DIM,
        skill_input_dim=skills_dim_in,
        skill_dim=SKILL_DIM,
    bidirectional=BIDIRECTIONAL,
).to(device)


# 4. Build test loader and accumulate predictions across checkpoints.
test_dataset_obj = CareerHistoryDataset(
    test_history_embeddings, test_history_months,
    test_history_skills,
    test_summary_embeddings, test_esco_embeddings
)
test_loader = DataLoader(
    test_dataset_obj, batch_size=batch_size, shuffle=False, collate_fn=collate_pad
)

sum_preds = None
per_seed_chunks = []
for ckpt in ckpt_paths:
    m = _build_test_model()
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    chunk = []
    with torch.no_grad():
        for (padded, padded_months, batch_skills,
             summaries, lengths, _) in test_loader:
            padded        = padded.to(device)
            padded_months = padded_months.to(device)
            batch_skills  = batch_skills.to(device)
            summaries     = summaries.to(device)
            preds = m(padded, padded_months,
                      batch_skills, summaries, lengths)
            chunk.append(F.normalize(preds, dim=-1).cpu())
    chunk = torch.cat(chunk, dim=0)
    per_seed_chunks.append(chunk)
    sum_preds = chunk if sum_preds is None else sum_preds + chunk

avg_preds_np = F.normalize(sum_preds / len(ckpt_paths), dim=-1).numpy().astype("float32")

# 5. Top-10 nearest labels.
_, top_k_indices = index.search(avg_preds_np, 10)

ground_truth_ids = [text_to_label_id[target] for *_, target in pairs["test"]]
predictions = [
    (true_id, top_k_indices[i].tolist())
    for i, true_id in enumerate(ground_truth_ids)
]


def mrr(preds_list):
    if not preds_list:
        return float("nan")
    ranks = []
    for true_id, preds in preds_list:
        if true_id in preds:
            ranks.append(1 / (preds.index(true_id) + 1))
        else:
            ranks.append(0)
    return sum(ranks) / len(preds_list)


def r_at_k(preds_list, k):
    if not preds_list:
        return float("nan")
    hits = sum(1 for true_id, preds in preds_list if true_id in preds[:k])
    return hits / len(preds_list)


def bucket_scores(preds_list, idxs):
    sub = [preds_list[i] for i in idxs]
    return {
        "n":    len(sub),
        "MRR":  round(mrr(sub),       4),
        "R@5":  round(r_at_k(sub, 5), 4),
        "R@10": round(r_at_k(sub, 10), 4),
    }


# Per-prefix-length buckets.
test_lens = [len(p[0]) for p in pairs["test"]]
idx_short = [i for i, L in enumerate(test_lens) if L == 1]
idx_long  = [i for i, L in enumerate(test_lens) if L >= 2]

scores = {
    "MRR":  round(mrr(predictions),       4),
    "R@5":  round(r_at_k(predictions, 5), 4),
    "R@10": round(r_at_k(predictions, 10), 4),
    "n_models": len(ckpt_paths),
    "by_prefix_len": {
        "1":   bucket_scores(predictions, idx_short),
        ">=2": bucket_scores(predictions, idx_long),
    },
}
print(scores)

# Per-seed test metrics (for std across seeds).
import statistics
per_seed_scores = []
for ckpt, chunk in zip(ckpt_paths, per_seed_chunks):
    seed_preds_np = chunk.numpy().astype("float32")
    _, seed_top_k = index.search(seed_preds_np, 10)
    seed_predictions = [
        (text_to_label_id[target], seed_top_k[j].tolist())
        for j, (*_, target) in enumerate(pairs["test"])
    ]
    per_seed_scores.append({
        "ckpt": os.path.basename(ckpt),
        "MRR":  round(mrr(seed_predictions),       4),
        "R@5":  round(r_at_k(seed_predictions, 5), 4),
        "R@10": round(r_at_k(seed_predictions, 10), 4),
    })
scores["per_seed"] = per_seed_scores
if len(per_seed_scores) >= 2:
    for _metric in ("MRR", "R@5", "R@10"):
        _vals = [s[_metric] for s in per_seed_scores]
        scores[f"{_metric}_seed_mean"] = round(statistics.mean(_vals),  4)
        scores[f"{_metric}_seed_std"]  = round(statistics.stdev(_vals), 4)
print("per-seed test scores:")
for s in per_seed_scores:
    print(f"  {s['ckpt']:40s}  MRR={s['MRR']:.4f}  R@5={s['R@5']:.4f}  R@10={s['R@10']:.4f}")
if len(per_seed_scores) >= 2:
    for _metric in ("MRR", "R@5", "R@10"):
        print(f"seed mean/std  {_metric}={scores[f'{_metric}_seed_mean']:.4f} +/- {scores[f'{_metric}_seed_std']:.4f}")

with open(SCORES_PATH, "w") as f:
    json.dump(scores, f, indent=4)
with open(PREDICTIONS_PATH, "wb") as f:
    pickle.dump(predictions, f)

print(f"scores saved to      {SCORES_PATH}")
print(f"predictions saved to {PREDICTIONS_PATH}")


Embedding career history, summaries, and ESCO targets...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.26it/s]


using ENSEMBLE of 10 checkpoints
{'MRR': 0.2948, 'R@5': 0.4214, 'R@10': 0.5132, 'n_models': 10, 'by_prefix_len': {'1': {'n': 569, 'MRR': 0.2878, 'R@5': 0.4095, 'R@10': 0.4974}, '>=2': {'n': 1218, 'MRR': 0.298, 'R@5': 0.4269, 'R@10': 0.5205}}}
per-seed test scores:
  gru_model_decorte_v2_1_seed0.pth          MRR=0.2935  R@5=0.4091  R@10=0.5092
  gru_model_decorte_v2_1_seed1.pth          MRR=0.2959  R@5=0.4197  R@10=0.5042
  gru_model_decorte_v2_1_seed2.pth          MRR=0.2893  R@5=0.4124  R@10=0.4980
  gru_model_decorte_v2_1_seed3.pth          MRR=0.2903  R@5=0.4079  R@10=0.5020
  gru_model_decorte_v2_1_seed4.pth          MRR=0.2970  R@5=0.4169  R@10=0.5008
  gru_model_decorte_v2_1_seed5.pth          MRR=0.2915  R@5=0.4247  R@10=0.5076
  gru_model_decorte_v2_1_seed6.pth          MRR=0.2931  R@5=0.4275  R@10=0.5104
  gru_model_decorte_v2_1_seed7.pth          MRR=0.2947  R@5=0.4158  R@10=0.5064
  gru_model_decorte_v2_1_seed8.pth          MRR=0.2955  R@5=0.4242  R@10=0.5187
  gru_model_dec